Neural representations, gradients, and output layers

Problem specification and design

X = {(0,0),(0,1),(1,0),(1,1)}, Y = {0,1}, labels = [0,1,1,0]. The positive examples occupy opposite corners of the square, so no single straight boundary separates them from both negative corners. A single affine layer followed by sigmoid cannot classify all four points correctly. Stacking affine layers still produces an affine map; nonlinear hidden activation is necessary.

The binary network has two inputs, two sigmoid hidden units, and one output logit. BCEWithLogitsLoss combines sigmoid and binary cross-entropy stably. Full-batch SGD uses learning rate 0.5 for 5000 CPU steps with seed 0. The same architecture, seed, rate, and steps are used for the activation comparison. Successful learning requires lower loss, all four correct labels, an early nonzero gradient, and repeatable results under the same seed. Hidden units have no direct labels; backpropagation supplies each unit's contribution to the output-loss gradient.

The forward pass is SensorNetwork.forward. train computes a scalar criterion, loss.backward performs reverse-mode differentiation, and optimizer.step changes parameters. The first-layer gradient is captured before the first update, matching the requested early measurement. Final loss and gradient are recomputed after the final update so they describe the final parameters.

Binary measurements

Initial loss: 0.69765902. Final loss: 0.03893126.

Probabilities: [[0.04311620816588402], [0.9676052331924438], [0.9676044583320618], [0.04475636035203934]]. Labels: [0, 1, 1, 0]. All correct: True. Same-seed repeat identical: True.

Early first-layer gradient: [[-6.063847104087472e-05, -5.5635900935158134e-05], [-0.0005683740600943565, -0.0006810622289776802]]. Final gradient: [[-0.001993275247514248, -0.001987171359360218], [0.001239426201209426, 0.0012338922824710608]].

parameter.grad is dL/dW for that parameter. With mean reduction over four examples it is the sum of the example gradients divided by four. A nonzero gradient alone is insufficient: falling loss and correct predictions demonstrate that the updates improve the task.

Activation comparison

| Hidden activation | Final loss | All correct | First-step gradient norm |
|---|---:|---:|---:|
| sigmoid | 0.03893126 | True | 0.00089088 |
| tanh | 0.00133684 | True | 0.06178480 |
| relu | 0.47741830 | False | 0.00169880 |

Sigmoid and tanh both learn XOR in these runs. ReLU fails with the fixed comparison settings; its result is retained rather than changing the architecture or hiding the failure. Sigmoid saturation gives small derivatives at large pre-activation magnitude; inactive ReLU units have exactly zero derivatives for negative pre-activations. Inspecting pre-activations and hidden outputs distinguishes these mechanisms. These four points and one seed do not establish a universally best activation.

Zero-initialization experiment

All parameters, including biases, are zeroed. The saved snapshots at steps 1, 2, 3, 6, 10, and 100 show identical hidden rows. In this balanced XOR task all outputs remain 0.5, loss stays near log(2), and gradients vanish at this initialization. Identical hidden units cannot develop different features because they compute identical outputs and receive identical signals. Random initialization breaks this symmetry.

Three-class extension

The labels are [0,1,1,2]. The hidden layer remains two sigmoid units, while the output produces three logits and CrossEntropyLoss uses integer class labels. PyTorch stores the output weights as (out_features, in_features), so the matrix is 3 by 2. Each example has three logits. Softmax exponentiates and normalizes them, so the probabilities sum to one.

Initial loss: 1.07349825; final loss: 0.00318469; classes: [0, 1, 1, 2]; all correct: True.

| Input | Class probabilities | Prediction |
|---|---|---:|
| (0, 0) | [0.9966387748718262, 0.0033611466642469168, 6.918003236933146e-08] | 0 |
| (0, 1) | [0.0013877706369385123, 0.9971740245819092, 0.0014382431982085109] | 1 |
| (1, 0) | [0.0013932866277173162, 0.9971659779548645, 0.0014407355338335037] | 1 |
| (1, 1) | [2.9493389064327857e-08, 0.003697002772241831, 0.9963029623031616] | 2 |

Probability row sums: [1.0, 1.0, 1.0, 1.0]. Largest change after adding 100 to every logit: 7.799826562e-09. Autodiff versus p-y maximum error: 2.328306437e-10.

For one example, L = -z_y + log(sum(exp(z))), so dL/dz = p - one_hot(y). The code uses autodiff to check this formula on a single-example loss. For a mean batch loss, the corresponding per-example logit gradient is divided by batch size. A shared logit shift cancels from numerator and denominator. Subtracting the maximum logit therefore preserves probabilities while avoiding overflow.

Reflection

Depth without nonlinearities cannot solve XOR. Useful backpropagation is evidenced by reduced loss and correct predictions, while zero initialization demonstrates that an architecture with sufficient capacity can still fail to learn. Activation changes affect hidden geometry and local derivatives; the measured early gradient norms above are experimental observations, not universal claims. Output and loss must match the target type: binary logits pair with binary cross-entropy, while mutually exclusive classes pair with multiclass cross-entropy. The same softmax and p-y mathematics applies to next-token prediction over large vocabularies, although representation, context handling, computation, and data scale become much larger.

Codex improved productivity by supplying training and measurement code. Execution was essential to confirm classification, and inspection corrected the reference's final-gradient measurement and weight-shape explanation. In a larger model, keep loss monitoring, representative prediction checks, gradient norms, seed controls, and probability normalization checks. Exhaustive prediction tests and finite-difference checks over every parameter become too costly; sample selected parameters and examples instead. The recorded repeat establishes determinism for this machine and configuration, not convergence for every random seed or platform.

In [1]:
import torch
from torch import nn
from torch.nn import functional as functional


INPUTS = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
BINARY_TARGETS = torch.tensor([[0.], [1.], [1.], [0.]])
CLASS_TARGETS = torch.tensor([0, 1, 1, 2])


class SensorNetwork(nn.Module):
    def __init__(self, activation='sigmoid', outputs=1):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.output = nn.Linear(2, outputs)
        self.activation = {'sigmoid': nn.Sigmoid, 'tanh': nn.Tanh, 'relu': nn.ReLU}[activation]()

    def forward(self, inputs):
        return self.output(self.activation(self.hidden(inputs)))


def train(activation='sigmoid', outputs=1, seed=0, steps=5000, learning_rate=0.5, zero=False):
    torch.manual_seed(seed)
    network = SensorNetwork(activation, outputs)
    if zero:
        with torch.no_grad():
            for parameter in network.parameters():
                parameter.zero_()
    targets = BINARY_TARGETS if outputs == 1 else CLASS_TARGETS
    criterion = nn.BCEWithLogitsLoss() if outputs == 1 else nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(network.parameters(), lr=learning_rate)
    initial_loss = criterion(network(INPUTS), targets).item()
    snapshots = []
    early_gradient = None
    history = []
    for step in range(steps):
        optimizer.zero_grad()
        logits = network(INPUTS)
        loss = criterion(logits, targets)
        loss.backward()
        if step == 0:
            early_gradient = network.hidden.weight.grad.detach().clone()
        optimizer.step()
        if step in {0, 1, 2, 5, 9, 99, 999, steps - 1}:
            history.append({'step': step + 1, 'loss_before_update': loss.item()})
            if zero:
                snapshots.append({'step': step + 1, 'weights': network.hidden.weight.detach().tolist()})
    optimizer.zero_grad()
    final_logits = network(INPUTS)
    final_loss = criterion(final_logits, targets)
    final_loss.backward()
    final_gradient = network.hidden.weight.grad.detach().clone()
    probabilities = (final_logits.sigmoid() if outputs == 1 else final_logits.softmax(dim=1)).detach()
    predictions = ((probabilities >= 0.5).long().flatten() if outputs == 1 else probabilities.argmax(dim=1))
    result = {'initial_loss': initial_loss, 'final_loss': final_loss.item(),
              'probabilities': probabilities.tolist(), 'predictions': predictions.tolist(),
              'targets': targets.flatten().long().tolist(),
              'correct': bool(torch.equal(predictions, targets.flatten().long())),
              'early_gradient_norm': early_gradient.norm().item(),
              'early_gradient': early_gradient.tolist(), 'final_gradient': final_gradient.tolist(),
              'weight_snapshots': snapshots, 'loss_history': history,
              'output_weight_shape': list(network.output.weight.shape)}
    if outputs == 3:
        probe = final_logits[0].detach().clone().requires_grad_()
        probe_loss = functional.cross_entropy(probe.unsqueeze(0), CLASS_TARGETS[:1])
        actual = torch.autograd.grad(probe_loss, probe)[0]
        expected = probe.softmax(dim=0) - functional.one_hot(CLASS_TARGETS[0], 3)
        result['row_sums'] = probabilities.sum(dim=1).tolist()
        result['shift_error'] = (functional.softmax(final_logits.detach() + 100, dim=1) - probabilities).abs().max().item()
        result['logit_gradient'] = actual.tolist()
        result['gradient_formula_error'] = (actual - expected).abs().max().item()
    return result


def experiment():
    torch.set_num_threads(1)
    binary = train()
    repeat = train()
    return {'binary': binary, 'repeat_identical': binary == repeat,
            'activations': {name: (binary if name == 'sigmoid' else train(name)) for name in ('sigmoid', 'tanh', 'relu')},
            'zero_initialization': train(zero=True, steps=100),
            'multiclass': train(outputs=3)}

import json
results = experiment()
print(json.dumps(results, indent=2))


{
  "binary": {
    "initial_loss": 0.6976590156555176,
    "final_loss": 0.03893125802278519,
    "probabilities": [
      [
        0.04311620816588402
      ],
      [
        0.9676052331924438
      ],
      [
        0.9676044583320618
      ],
      [
        0.04475636035203934
      ]
    ],
    "predictions": [
      0,
      1,
      1,
      0
    ],
    "targets": [
      0,
      1,
      1,
      0
    ],
    "correct": true,
    "early_gradient_norm": 0.0008908800664357841,
    "early_gradient": [
      [
        -6.063847104087472e-05,
        -5.5635900935158134e-05
      ],
      [
        -0.0005683740600943565,
        -0.0006810622289776802
      ]
    ],
    "final_gradient": [
      [
        -0.001993275247514248,
        -0.001987171359360218
      ],
      [
        0.001239426201209426,
        0.0012338922824710608
      ]
    ],
    "weight_snapshots": [],
    "loss_history": [
      {
        "step": 1,
        "loss_before_update": 0.6976590156555176
   

In [2]:
assert results['binary']['correct']
assert results['binary']['final_loss'] < results['binary']['initial_loss']
assert results['binary']['early_gradient_norm'] > 0
assert results['repeat_identical']
assert all(row['weights'][0] == row['weights'][1] for row in results['zero_initialization']['weight_snapshots'])
assert not results['zero_initialization']['correct']
assert results['multiclass']['correct']
assert results['multiclass']['output_weight_shape'] == [3, 2]
assert all(abs(total - 1) < 1e-6 for total in results['multiclass']['row_sums'])
assert results['multiclass']['shift_error'] < 1e-5
assert results['multiclass']['gradient_formula_error'] < 1e-6
print('Learning, gradients, symmetry, and output checks passed')


Learning, gradients, symmetry, and output checks passed
